In [1]:
import torch
import ultralytics

print("--- SYSTEM DIAGNOSTICS ---")
print(f"PyTorch Version: {torch.__version__}")
print(f"Ultralytics Version: {ultralytics.__version__}\n")

cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("\n⚠️ WARNING: CUDA is NOT available!")
    print("Your models will train on the CPU, which is extremely slow.")

--- SYSTEM DIAGNOSTICS ---
PyTorch Version: 2.7.1+cu118
Ultralytics Version: 8.4.37

CUDA Available: True
GPU Device Name: NVIDIA GeForce RTX 3060
Number of GPUs: 1


In [3]:
import os

# --- UPDATE THIS TO YOUR MAIN RAF-DB FOLDER ---
# (The folder that contains the 'train' and 'test' subfolders)
DATASET_DIR = r'C:\Users\User\Desktop\AiAssignment\Ai-Assignment\RAF-DB'
# ----------------------------------------------

emotion_map = {
    '1': 'surprise', '2': 'fear', '3': 'disgust', 
    '4': 'happy', '5': 'sad', '6': 'angry', '7': 'neutral'
}

print("Renaming folders...")

for split in ['train', 'test']:
    split_dir = os.path.join(DATASET_DIR, split)
    
    # Check if the train/test folder exists
    if not os.path.exists(split_dir):
        print(f"⚠️ Could not find folder: {split_dir}")
        continue

    for folder_num, emotion_name in emotion_map.items():
        old_path = os.path.join(split_dir, folder_num)
        new_path = os.path.join(split_dir, emotion_name)
        
        # If the numbered folder exists, rename it
        if os.path.exists(old_path):
            os.rename(old_path, new_path)
            print(f"✅ Renamed: {split}/{folder_num} -> {split}/{emotion_name}")
        elif os.path.exists(new_path):
            print(f"ℹ️ Already named correctly: {split}/{emotion_name}")

print("\n🚀 Phase 1 Complete! Move straight to the Phase 2 Preprocessing script.")

Renaming folders...
✅ Renamed: train/1 -> train/surprise
✅ Renamed: train/2 -> train/fear
✅ Renamed: train/3 -> train/disgust
✅ Renamed: train/4 -> train/happy
✅ Renamed: train/5 -> train/sad
✅ Renamed: train/6 -> train/angry
✅ Renamed: train/7 -> train/neutral
✅ Renamed: test/1 -> test/surprise
✅ Renamed: test/2 -> test/fear
✅ Renamed: test/3 -> test/disgust
✅ Renamed: test/4 -> test/happy
✅ Renamed: test/5 -> test/sad
✅ Renamed: test/6 -> test/angry
✅ Renamed: test/7 -> test/neutral

🚀 Phase 1 Complete! Move straight to the Phase 2 Preprocessing script.


In [6]:
import os
import cv2
import numpy as np
import glob

# --- UPDATE THIS PATH TO YOUR NEW RAF-DB TRAIN FOLDER ---
TRAIN_DIR = r'C:\Users\User\Desktop\AiAssignment\Ai-Assignment\RAF-DB\train'
# --------------------------------------------------------

def preprocess_and_augment(img):
    """Applies noise reduction and random augmentation."""
    # 1. Noise Reduction (Gaussian Blur)
    blurred = cv2.GaussianBlur(img, (3, 3), 0)
    
    # 2. Random Horizontal Flip
    if np.random.rand() > 0.5:
        blurred = cv2.flip(blurred, 1)
        
    # 3. Random Brightness (Simulates different lighting) safely
    value = np.random.randint(-20, 20)
    
    # Cast to int32 to safely do math beyond the 0-255 limits
    blurred_int = blurred.astype(np.int32)
    blurred_int = blurred_int + value
    
    # Force everything back into the 0-255 range
    blurred_clipped = np.clip(blurred_int, 0, 255)
    
    return blurred_clipped.astype(np.uint8)

emotions = os.listdir(TRAIN_DIR)
class_counts = {}

# Step 1: Find the target count (the size of the largest class)
for emotion in emotions:
    folder_path = os.path.join(TRAIN_DIR, emotion)
    if os.path.isdir(folder_path):
        class_counts[emotion] = len(glob.glob(os.path.join(folder_path, '*.jpg')))

TARGET_COUNT = max(class_counts.values())
print(f"🎯 Target balance count set to: {TARGET_COUNT} images per class")

# Step 2: Augment minority classes
for emotion in emotions:
    folder_path = os.path.join(TRAIN_DIR, emotion)
    if not os.path.isdir(folder_path): continue
        
    images = glob.glob(os.path.join(folder_path, '*.jpg'))
    current_count = len(images)
    shortfall = TARGET_COUNT - current_count
    
    if shortfall > 0:
        print(f" -> {emotion.upper()}: Generating {shortfall} synthetic images...")
        generated = 0
        while generated < shortfall:
            random_img_path = np.random.choice(images)
            img = cv2.imread(random_img_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None: continue
            
            # Ensure 100x100 resolution
            img = cv2.resize(img, (100, 100))
            final_img = preprocess_and_augment(img)
            
            new_filename = f"aug_{generated}_{os.path.basename(random_img_path)}"
            cv2.imwrite(os.path.join(folder_path, new_filename), final_img)
            generated += 1
            
    # Step 3: Standardize the original images (Grayscale & 100x100)
    print(f" -> {emotion.upper()}: Standardizing original images...")
    for img_path in images:
        img = cv2.imread(img_path)
        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            resized = cv2.resize(gray, (100, 100))
            cv2.imwrite(img_path, resized)

print("\n✅ RAF-DB Preprocessing Complete! Ready for Training.")

🎯 Target balance count set to: 4772 images per class
 -> ANGRY: Generating 4067 synthetic images...
 -> ANGRY: Standardizing original images...
 -> DISGUST: Generating 4055 synthetic images...
 -> DISGUST: Standardizing original images...
 -> FEAR: Generating 4491 synthetic images...
 -> FEAR: Standardizing original images...
 -> HAPPY: Standardizing original images...
 -> NEUTRAL: Generating 2248 synthetic images...
 -> NEUTRAL: Standardizing original images...
 -> SAD: Generating 2790 synthetic images...
 -> SAD: Standardizing original images...
 -> SURPRISE: Generating 3482 synthetic images...
 -> SURPRISE: Standardizing original images...

✅ RAF-DB Preprocessing Complete! Ready for Training.
